In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = sns.load_dataset("titanic")

In [13]:
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
data["is_female"] = (data["sex"] == "female").astype(int)
X = data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]]
y = data["survived"]
print(X.shape, round(y.mean(), 2))
print(X.isna().sum().to_dict())

(891, 6) 0.38
{'pclass': 0, 'is_female': 0, 'age': 177, 'fare': 0, 'sibsp': 0, 'parch': 0}


- Column X with data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]] are features.
- Column y with data["survived"] is the label.
- I must not use alive as a feature, because it is the label written in words (survived 1 = alive “yes”, 0 = “no”). That is target leakage: the model would read the answer instead of learning, and for a new passenger this information does not exist yet.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))
print(X_train["age"].isna().sum())
print(X_test["age"].isna().sum())

(712, 6) (179, 6) 0.383 0.385
137
40


- To ensure the model has actually learned to generalise to brand-new, unseen situations rather than just memorising the training examples.

In [15]:
base = DummyClassifier(strategy="most_frequent")
base.fit(X_train, y_train)
print(base.score(X_test, y_test))

0.6145251396648045


- It tells that a model that learns nothing already gets 61.45%. So any real model must beat 61.45% to be useful.

In [ ]:
age_median = X_train["age"].median()
X_train_f = X_train.copy()
X_test_f = X_test.copy()
X_train_f["age"] = X_train_f["age"].fillna(age_median)
X_test_f["age"] = X_test_f["age"].fillna(age_median)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_f, y_train)
print(model.score(X_train_f, y_train))
print(model.score(X_test_f, y_test))

0.7949438202247191
0.8044692737430168
28.5


- Yes. The model scores 0.8045 on the test set, against 0.6145 for the baseline, which is 19.0 percentage points better. The train score (0.7949) is close to the test score, so the model is not just memorising the training data.

In [19]:
X_wrong = X.copy()
X_wrong["age"] = X_wrong["age"].fillna(X_wrong["age"].median())
a, b, c, d = train_test_split(X_wrong, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000)
model.fit(a, c)
print("Wrong:", round(model.score(a, c), 4), round(model.score(b, d), 4))

Wrong: 0.7921 0.8045


- It is still wrong because the median 28.0 was computed from all 891 rows, which includes the test passengers’ ages. Test information leaked into the cleaning step before the split. The scores are almost the same (train 0.7921 vs 0.7949, test 0.8045 for both) because one median barely changes anything, but the rule matters: every number used for cleaning must come from the training data only.

In [26]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
print("Pipe:", round(pipe.score(X_train, y_train), 3), round(pipe.score(X_test, y_test), 3))
print(pipe.named_steps["imputer"].statistics_.round(2))

Pipe: 0.795 0.804
[ 3.    0.   28.5  14.45  0.    0.  ]


- The pipeline fits the imputer where it reads X_train's data, finds the median and stores into statistics_
- It does clean the data and fill in any missing value in X_test data with X_train's median
- It passes this freshly cleaned, fully numeric dataset straight into LogisticRegression model to calculate the optimal weights.
- Because the imputer is fitted inside the pipeline on X_train only, I cannot make the Part 5 mistake by accident. The same steps run automatically when I call .score on any data, and the scores match Part 4 (0.7949 and 0.8045) without any manual filling.

In [33]:
X_leak = X.copy()
X_leak["alive_yes"] = (df["alive"] == "yes").astype(int)
a, b, c, d = train_test_split(X_leak, y, test_size=0.2, random_state=42, stratify=y)
pl = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000)),
])
pl.fit(a, c)
print("Leak:", pl.score(b, d))


Leak: 1.0


- A score of 1.0 is too good to be true: the honest model scored 0.8045. The leaked column is alive_yes, which is the label survived written in another form (1 when “yes”, 0 when “no”). The model copies the answer instead of learning, and in real use this column would not exist yet for a new passenger. When a score is too good to be true, I look for leakage before celebrating.

## Findings
1. The baseline achieved 61.45% where it really learns nothing, because the basseline learns with strategy="most_frequent". Tt predicts “did not survive” for everyone, and 61.45% of test passengers did not survive.

2. The model leaks with 1.0 (100%) because the alive_yes is an exact copy of the label (1 for “yes”, 0 for “no”). The score jumped from 0.8045 to 1.0.

3. When the train and test dataset's missing values filled with the column's median(28.5), the score actually gets 19 points better than the 61.45% baseline which is 79.49% and 80.45% for train and test respectively.

## Leakage Rule
- If an input feature contains information that is not available at the moment, it must be removed to 
misleadingly perfect score.

## Caveat
- One test set of 179 passengers is a small sample, so the 80.45% is an estimate not a guarantee.